# Notebook 4B: Encode Discharge and Radiology Notes with BioClinical-ModernBERT

## Objective
Encode both discharge summaries and chest X-ray reports using BioClinical-ModernBERT, a transformer model with 8192 token context window specifically trained on biomedical and clinical text. No chunking required.

## Input Data
- `labeled_admissions_with_discharge_and_radiology.csv` (140,685 admissions from Notebook 2B)

## Model: BioClinical-ModernBERT
- **Model:** thomas-sounack/bioclinical-modernbert-base
- **Context length:** 8192 tokens (vs BERT's 512)
- **Architecture:** ModernBERT with clinical/biomedical pre-training
- **No chunking needed:** 99.7% of notes fit in single forward pass

## Process
Load BioClinical-ModernBERT model, encode discharge notes separately from radiology notes (two forward passes per admission), extract embeddings from CLS token or mean pooling, save both embedding sets for multimodal fusion.

## Output
- `discharge_embeddings_140k.npz` - Discharge note embeddings (140,685 × hidden_dim)
- `radiology_embeddings_140k.npz` - Radiology report embeddings (140,685 × hidden_dim)
- `embedding_hadm_id_mapping_140k.csv` - Maps embeddings to admission IDs

## Key Design Decisions
**Separate encoding:** Encode discharge and radiology independently rather than concatenating text, allowing flexibility in fusion strategies (concatenate, average, attention) and enabling ablation studies (discharge-only vs radiology-only vs combined).

**Truncation strategy:** Simple truncation at 8192 tokens for 354 long notes (0.3%), keeping beginning of text where key clinical information typically appears.

**Embedding extraction:** Use CLS token embedding as document representation, standard practice for transformer-based text encoding.

## Expected Time
Approximately 2-3 hours on A100 GPU for 140K admissions with two forward passes each (280K total forward passes).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = torch.device('cuda')
else:
    print("WARNING: No GPU detected - encoding will be very slow")
    device = torch.device('cpu')

print(f"Using device: {device}")

PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 42.47 GB
Using device: cuda


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import gc
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
torch.set_grad_enabled(False)  # Inference mode

print("Libraries imported")

Libraries imported


In [5]:
DATA_DIR = Path('/content/drive/MyDrive/MIDS/w266/Final Project/data')
OUTPUT_DIR = Path('/content/drive/MyDrive/MIDS/w266/Final Project/output')
EMBEDDINGS_DIR = OUTPUT_DIR / 'embeddings'

# Create embeddings directory if it doesn't exist
EMBEDDINGS_DIR.mkdir(exist_ok=True, parents=True)

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Embeddings directory: {EMBEDDINGS_DIR}")

Data directory: /content/drive/MyDrive/MIDS/w266/Final Project/data
Output directory: /content/drive/MyDrive/MIDS/w266/Final Project/output
Embeddings directory: /content/drive/MyDrive/MIDS/w266/Final Project/output/embeddings


In [6]:
# Load the combined discharge + radiology dataset
print("Loading dataset from Notebook 2B...")

data_df = pd.read_csv(OUTPUT_DIR / 'labeled_admissions_with_discharge_and_radiology.csv')

print(f"Loaded {len(data_df):,} admissions")
print(f"\nColumns: {data_df.columns.tolist()}")
print(f"\nReadmission distribution:")
print(data_df['readmitted_30day'].value_counts())

Loading dataset from Notebook 2B...
Loaded 140,685 admissions

Columns: ['hadm_id', 'subject_id', 'admittime', 'dischtime', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'discharge_text', 'discharge_charttime', 'discharge_note_id', 'radiology_text', 'radiology_charttime', 'radiology_note_id', 'readmitted_30day']

Readmission distribution:
readmitted_30day
0    109644
1     31041
Name: count, dtype: int64


In [7]:
# Load tokenizer and model
print("\nLoading BioClinical-ModernBERT model...")
print("This may take a few minutes on first load...")

MODEL_NAME = "thomas-sounack/bioclinical-modernbert-base"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME)

    # Move model to GPU
    model = model.to(device)
    model.eval()  # Set to evaluation mode

    print(f"\n✓ Model loaded successfully: {MODEL_NAME}")
    print(f"✓ Model on device: {next(model.parameters()).device}")

    # Get model info
    print(f"\nModel configuration:")
    print(f"  Hidden size: {model.config.hidden_size}")
    print(f"  Max position embeddings: {model.config.max_position_embeddings}")
    print(f"  Number of layers: {model.config.num_hidden_layers}")
    print(f"  Number of attention heads: {model.config.num_attention_heads}")

except Exception as e:
    print(f"\n✗ Error loading model: {e}")
    print("\nTroubleshooting:")
    print("  1. Check model name is correct")
    print("  2. Check internet connection")
    print("  3. Try: !pip install --upgrade transformers")


Loading BioClinical-ModernBERT model...
This may take a few minutes on first load...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]


✓ Model loaded successfully: thomas-sounack/bioclinical-modernbert-base
✓ Model on device: cuda:0

Model configuration:
  Hidden size: 768
  Max position embeddings: 8192
  Number of layers: 22
  Number of attention heads: 12


In [8]:
# Test encoding on a sample
print("\nTesting model on sample discharge note...")

sample_text = data_df['discharge_text'].iloc[0][:1000]  # First 1000 chars
print(f"Sample text length: {len(sample_text)} characters")
print(f"First 300 characters:\n{sample_text[:300]}...\n")

# Tokenize
inputs = tokenizer(
    sample_text,
    max_length=8192,
    truncation=True,
    padding='max_length',
    return_tensors='pt'
).to(device)

print(f"Tokenized input shape: {inputs['input_ids'].shape}")

# Encode
with torch.no_grad():
    outputs = model(**inputs)
    embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # CLS token

print(f"Embedding shape: {embedding.shape}")
print(f"Embedding sample (first 10 values): {embedding[0, :10]}")

print("\n✓ Test successful - ready for batch encoding")


Testing model on sample discharge note...
Sample text length: 1000 characters
First 300 characters:
 
Name:  ___                     Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: MEDICINE
 
Allergies: 
Percocet
 
Attending: ___.
 
Chief Complaint:
abdominal fullness and discomfort
 
Major Surgical or Invasive Procedur...

Tokenized input shape: torch.Size([1, 8192])


/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:282: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


Embedding shape: (1, 768)
Embedding sample (first 10 values): [ 2.1970625   0.3513442   1.5155164  -1.2402462  -1.8507679   0.14162943
 -0.724527   -1.5814228   1.4989015  -0.62244856]

✓ Test successful - ready for batch encoding


In [9]:
def encode_texts(texts, model, tokenizer, device, batch_size=16, desc="Encoding"):
    """
    Encode a list of texts using the model.

    Args:
        texts: List of text strings
        model: Transformer model
        tokenizer: Tokenizer
        device: torch device
        batch_size: Batch size for encoding
        desc: Description for progress bar

    Returns:
        numpy array of shape (num_texts, hidden_dim)
    """
    embeddings = []

    # Process in batches
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch_texts = texts[i:i + batch_size]

        # Tokenize batch
        inputs = tokenizer(
            batch_texts,
            max_length=8192,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        ).to(device)

        # Encode
        with torch.no_grad():
            outputs = model(**inputs)
            # Use CLS token embedding (first token)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

        embeddings.append(batch_embeddings)

        # Free GPU memory periodically
        if (i // batch_size) % 100 == 0:
            torch.cuda.empty_cache()

    return np.vstack(embeddings)

print("Encoding function defined")

Encoding function defined


In [10]:
# Quick analysis of text lengths
print("\nAnalyzing text lengths before encoding...")

# Estimate tokens (rough: 1 token ≈ 4 characters)
data_df['discharge_tokens_est'] = data_df['discharge_text'].str.len() / 4
data_df['radiology_tokens_est'] = data_df['radiology_text'].str.len() / 4

print(f"\nDischarge note token estimates:")
print(f"  Mean: {data_df['discharge_tokens_est'].mean():.0f}")
print(f"  Median: {data_df['discharge_tokens_est'].median():.0f}")
print(f"  Max: {data_df['discharge_tokens_est'].max():.0f}")
print(f"  >8192 tokens: {(data_df['discharge_tokens_est'] > 8192).sum():,} ({(data_df['discharge_tokens_est'] > 8192).sum()/len(data_df)*100:.2f}%)")

print(f"\nRadiology note token estimates:")
print(f"  Mean: {data_df['radiology_tokens_est'].mean():.0f}")
print(f"  Median: {data_df['radiology_tokens_est'].median():.0f}")
print(f"  Max: {data_df['radiology_tokens_est'].max():.0f}")
print(f"  >8192 tokens: {(data_df['radiology_tokens_est'] > 8192).sum():,}")

print(f"\nTotal notes to encode: {len(data_df) * 2:,} (discharge + radiology)")


Analyzing text lengths before encoding...

Discharge note token estimates:
  Mean: 2905
  Median: 2733
  Max: 15095
  >8192 tokens: 304 (0.22%)

Radiology note token estimates:
  Mean: 146
  Median: 129
  Max: 2584
  >8192 tokens: 0

Total notes to encode: 281,370 (discharge + radiology)


In [11]:
# Encode all discharge notes
print("\n" + "="*60)
print("ENCODING DISCHARGE NOTES")
print("="*60)

# Get discharge texts as list
discharge_texts = data_df['discharge_text'].tolist()

print(f"\nEncoding {len(discharge_texts):,} discharge notes...")
print(f"Estimated time: ~1-2 hours on A100 GPU")
print(f"Batch size: 16")

# Encode
BATCH_SIZE = 16
discharge_embeddings = encode_texts(
    discharge_texts,
    model,
    tokenizer,
    device,
    batch_size=BATCH_SIZE,
    desc="Encoding discharge notes"
)

print(f"\n✓ Discharge encoding complete!")
print(f"Embeddings shape: {discharge_embeddings.shape}")

# Free memory
del discharge_texts
gc.collect()
torch.cuda.empty_cache()


ENCODING DISCHARGE NOTES

Encoding 140,685 discharge notes...
Estimated time: ~1-2 hours on A100 GPU
Batch size: 16


Encoding discharge notes:   5%|▍         | 435/8793 [43:37<13:58:04,  6.02s/it]


KeyboardInterrupt: 

In [ ]:
# Encode all radiology reports
print("\n" + "="*60)
print("ENCODING RADIOLOGY REPORTS")
print("="*60)

# Get radiology texts as list
radiology_texts = data_df['radiology_text'].tolist()

print(f"\nEncoding {len(radiology_texts):,} radiology reports...")
print(f"Estimated time: ~30-60 minutes on A100 GPU")
print(f"Batch size: 16")

# Encode
radiology_embeddings = encode_texts(
    radiology_texts,
    model,
    tokenizer,
    device,
    batch_size=BATCH_SIZE,
    desc="Encoding radiology reports"
)

print(f"\n✓ Radiology encoding complete!")
print(f"Embeddings shape: {radiology_embeddings.shape}")

# Free memory
del radiology_texts
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Verify embeddings look reasonable
print("\n" + "="*60)
print("VERIFYING EMBEDDINGS")
print("="*60)

print(f"\nDischarge embeddings:")
print(f"  Shape: {discharge_embeddings.shape}")
print(f"  Mean: {discharge_embeddings.mean():.4f}")
print(f"  Std: {discharge_embeddings.std():.4f}")
print(f"  Min: {discharge_embeddings.min():.4f}")
print(f"  Max: {discharge_embeddings.max():.4f}")

print(f"\nRadiology embeddings:")
print(f"  Shape: {radiology_embeddings.shape}")
print(f"  Mean: {radiology_embeddings.mean():.4f}")
print(f"  Std: {radiology_embeddings.std():.4f}")
print(f"  Min: {radiology_embeddings.min():.4f}")
print(f"  Max: {radiology_embeddings.max():.4f}")

# Check for NaN or Inf
print(f"\nQuality checks:")
print(f"  Discharge NaN: {np.isnan(discharge_embeddings).sum()}")
print(f"  Discharge Inf: {np.isinf(discharge_embeddings).sum()}")
print(f"  Radiology NaN: {np.isnan(radiology_embeddings).sum()}")
print(f"  Radiology Inf: {np.isinf(radiology_embeddings).sum()}")

if np.isnan(discharge_embeddings).sum() > 0 or np.isnan(radiology_embeddings).sum() > 0:
    print("\n⚠ WARNING: NaN values detected in embeddings!")

In [ ]:
# Save embeddings
print("\n" + "="*60)
print("SAVING EMBEDDINGS")
print("="*60)

# Get hadm_ids
hadm_ids = data_df['hadm_id'].values

# Save discharge embeddings
discharge_file = EMBEDDINGS_DIR / 'discharge_embeddings_140k.npz'
np.savez_compressed(
    discharge_file,
    embeddings=discharge_embeddings,
    hadm_ids=hadm_ids
)
print(f"\n✓ Saved discharge embeddings: {discharge_file}")
print(f"  File size: {discharge_file.stat().st_size / (1024*1024):.2f} MB")

# Save radiology embeddings
radiology_file = EMBEDDINGS_DIR / 'radiology_embeddings_140k.npz'
np.savez_compressed(
    radiology_file,
    embeddings=radiology_embeddings,
    hadm_ids=hadm_ids
)
print(f"✓ Saved radiology embeddings: {radiology_file}")
print(f"  File size: {radiology_file.stat().st_size / (1024*1024):.2f} MB")

# Save mapping
mapping_file = EMBEDDINGS_DIR / 'embedding_hadm_id_mapping_140k.csv'
mapping_df = pd.DataFrame({
    'hadm_id': hadm_ids,
    'embedding_index': range(len(hadm_ids))
})
mapping_df.to_csv(mapping_file, index=False)
print(f"✓ Saved mapping: {mapping_file}")

In [ ]:
# Load and verify
print("\n" + "="*60)
print("VERIFYING SAVED FILES")
print("="*60)

# Load discharge
loaded_discharge = np.load(discharge_file)
print(f"\nDischarge embeddings:")
print(f"  Loaded shape: {loaded_discharge['embeddings'].shape}")
print(f"  Loaded hadm_ids: {len(loaded_discharge['hadm_ids'])}")

# Load radiology
loaded_radiology = np.load(radiology_file)
print(f"\nRadiology embeddings:")
print(f"  Loaded shape: {loaded_radiology['embeddings'].shape}")
print(f"  Loaded hadm_ids: {len(loaded_radiology['hadm_ids'])}")

# Verify they match
assert np.array_equal(loaded_discharge['embeddings'], discharge_embeddings), "Discharge embeddings don't match!"
assert np.array_equal(loaded_radiology['embeddings'], radiology_embeddings), "Radiology embeddings don't match!"
assert np.array_equal(loaded_discharge['hadm_ids'], loaded_radiology['hadm_ids']), "hadm_ids don't match!"

print(f"\n✓ All verifications passed - files saved correctly")

In [ ]:
# Visualize embedding distributions
print("\nVisualizing embedding distributions...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Discharge embeddings
axes[0].hist(discharge_embeddings.flatten(), bins=100, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Embedding Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Discharge Embedding Distribution')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_yscale('log')

# Radiology embeddings
axes[1].hist(radiology_embeddings.flatten(), bins=100, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Embedding Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Radiology Embedding Distribution')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'embedding_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("Visualization saved")

In [ ]:
print("\n" + "="*60)
print("NOTEBOOK 4B COMPLETE: NOTES ENCODED")
print("="*60)

print(f"\nSummary:")
print(f"  Model: {MODEL_NAME}")
print(f"  Context window: 8192 tokens")
print(f"  Admissions encoded: {len(data_df):,}")
print(f"  Embedding dimension: {discharge_embeddings.shape[1]}")

print(f"\nOutputs:")
print(f"  Discharge: {discharge_file}")
print(f"  Radiology: {radiology_file}")
print(f"  Mapping: {mapping_file}")

print(f"\nFile sizes:")
print(f"  Discharge: {discharge_file.stat().st_size / (1024*1024):.2f} MB")
print(f"  Radiology: {radiology_file.stat().st_size / (1024*1024):.2f} MB")

print(f"\nNext steps:")
print(f"  Notebook 5B: Engineer tabular features (age, labs, meds, Hi-BEHRT)")
print(f"  Notebook 6B: Multimodal fusion (text + tabular)")

# Clean up GPU memory
del model
del tokenizer
torch.cuda.empty_cache()
gc.collect()
print("\n✓ GPU memory cleared")